# Rede neural convolucional com transformada Wavelet no pipeline 


## 1. Dataset - MiniMIAS

### 1.1 Rodar localmente o programa

- Para isso é necessário baixar e extrair o dataset para a máquina local. 

**Link:** https://www.repository.cam.ac.uk/items/b6a97f0c-3b9b-40ad-8f18-3d121eef1459

- Baixe o dataset em Files;
- Após extrair em uma pasta os arquivos. Comando via terminal: _unzip MIAS-DB.zip_

In [5]:
import os

pasta = "dataset-mias/"

arquivos = os.listdir(pasta)
contador = 0

for arquivo in arquivos:
    if ".pgm" in arquivo:
        contador += 1

print(contador)

322


### 1.2 Classificando as imagnes

In [1]:
import os
import random
import shutil
from PyPDF2 import PdfReader

pasta_imagens = "dataset-mias/"

com_nodulo = []
sem_nodulo = []

# LER PDF
reader = PdfReader("dataset-mias/00README.pdf")
linhas = []

for pagina in reader.pages:
    texto = pagina.extract_text()
    linhas.extend(texto.split("\n"))

# PROCESSAR
for linha in linhas:
    if "mdb" in linha:
        partes = linha.split()
        nome = partes[0] + ".pgm"
        
        if "NORM" in linha:
            sem_nodulo.append(nome)
        else:
            com_nodulo.append(nome)

# DIVIDIR
def dividir(lista, pasta_train, pasta_test):
    random.shuffle(lista)
    corte = int(0.7 * len(lista))
    
    for nome in lista[:corte]:
        origem = os.path.join(pasta_imagens, nome)
        destino = os.path.join(pasta_train, nome)
        if os.path.exists(origem):
            shutil.copy(origem, destino)
    
    for nome in lista[corte:]:
        origem = os.path.join(pasta_imagens, nome)
        destino = os.path.join(pasta_test, nome)
        if os.path.exists(origem):
            shutil.copy(origem, destino)

# PASTAS
os.makedirs("dataset/train/com_nodulo", exist_ok=True)
os.makedirs("dataset/train/sem_nodulo", exist_ok=True)
os.makedirs("dataset/test/com_nodulo", exist_ok=True)
os.makedirs("dataset/test/sem_nodulo", exist_ok=True)

# EXECUTAR
dividir(com_nodulo, "dataset/train/com_nodulo", "dataset/test/com_nodulo")
dividir(sem_nodulo, "dataset/train/sem_nodulo", "dataset/test/sem_nodulo")

print("Finalizado!")

Finalizado!


### 1.3 Carregando dataset para CNN

In [2]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  
    transforms.Resize((28, 28)), 
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(
    root="dataset/train",
    transform=transform
)

test_dataset = datasets.ImageFolder(
    root="dataset/test",
    transform=transform
)

# Criar DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

# Verificar Classes
print(train_dataset.classes)

['com_nodulo', 'sem_nodulo']


## 2. Criando uma camada Wavelet usando PyTorch

In [3]:
# Camada Wavelet 
import torch
import torch.nn as nn
import torch.nn.functional as F

class WaveletLayer(nn.Module):
    def __init__(self):
        super(WaveletLayer, self).__init__()

        # Filtro Haar LL
        kernel = torch.tensor([[0.5, 0.5],
                               [0.5, 0.5]], dtype=torch.float32)

        # Ajustar formato: (out_channels, in_channels, H, W)
        kernel = kernel.view(1, 1, 2, 2)

        # Registrar como peso fixo (não treinável)
        self.register_buffer("weight", kernel)
        # para tornar a wavelet treinável pela própria cnn: self.weight = nn.Parameter(kernel)

    def forward(self, x):
        return F.conv2d(x, self.weight, stride=2)


## 3. Modelo com 2 convoluções de CNN integrando a camada Wavelet

<img src="img/waveletCNN-modelo2.png" width="700" title="Dica de ferramenta">

In [ ]:
"""
MODELO COM 2 CONVOLUÇÕES E APLICANDO CAMADA WAVELET 
"""
class WaveletCNN(nn.Module):
    def __init__(self):
        super(WaveletCNN, self).__init__()
        
        self.wavelet = WaveletLayer() # camada wavelet
        
        # Bloco 1
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32) # pega a saída de uma camada de convolução e a "normaliza" garantindo que a média seja próxima de 0 e o desvio padrão seja 1
        
        # Bloco 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        # Redução da dimensão
        self.pool = nn.MaxPool2d(2, 2)
        
        # Camada de decisão 
        # Entrada era 28x28 -> Wavelet(14x14) -> Pool(7x7)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.dropout = nn.Dropout(0.5)  # "desliga" aleatoriamente uma porcentagem dos neurônios
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x):
        x = self.wavelet(x)               # Saída 14x14
        
        x = torch.relu(self.bn1(self.conv1(x)))
        x = torch.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)                  # Saída 7x7
        
        x = x.view(x.size(0), -1) 
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

### 3.1 Treinando o modelo

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim

model = WaveletCNN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(50):
    model.train()
    
    for imagens, labels in train_loader:
        outputs = model(imagens)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch} - Loss: {loss.item()}")

Epoch 0 - Loss: 0.8104053139686584
Epoch 1 - Loss: 0.5911598801612854
Epoch 2 - Loss: 0.6736733317375183
Epoch 3 - Loss: 0.7536495327949524
Epoch 4 - Loss: 0.5407925248146057
Epoch 5 - Loss: 0.5404188632965088
Epoch 6 - Loss: 0.7270596623420715
Epoch 7 - Loss: 0.6612396240234375
Epoch 8 - Loss: 0.5908434987068176
Epoch 9 - Loss: 0.6052066087722778
Epoch 10 - Loss: 0.5786821842193604
Epoch 11 - Loss: 0.5736437439918518
Epoch 12 - Loss: 0.5570746660232544
Epoch 13 - Loss: 0.6815726161003113
Epoch 14 - Loss: 0.5880903601646423
Epoch 15 - Loss: 0.5106891989707947
Epoch 16 - Loss: 0.6083828210830688
Epoch 17 - Loss: 0.5566092729568481
Epoch 18 - Loss: 0.5687573552131653
Epoch 19 - Loss: 0.4942847788333893
Epoch 20 - Loss: 0.4342668652534485
Epoch 21 - Loss: 0.5870268940925598
Epoch 22 - Loss: 0.2624649405479431
Epoch 23 - Loss: 0.525729775428772
Epoch 24 - Loss: 0.49894455075263977
Epoch 25 - Loss: 0.43457379937171936
Epoch 26 - Loss: 0.4418482780456543
Epoch 27 - Loss: 0.5509337186813354
E

### 3.2 Testando o modelo

In [9]:
model.eval()
corretos = 0
total = 0

with torch.no_grad():
    for imagens, labels in test_loader:
        outputs = model(imagens)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        corretos += (predicted == labels).sum().item()

print(f"Acurácia: {100 * corretos / total:.2f}%")

Acurácia: 53.06%
